# GwenLand glcuda Wave 124 - stacked SiLU row-CTA

Production T4 A/B of the retained Wave 123 path against one CTA per token row.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave124-stacked-silu-rowcta-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "c0e325b2a754c088f88c652b76c3adb5364bf624"
PATCH_SHA256 = "5a3e304c1328ddf00f4194760c1a206ba7c75ef19ca3b0f52371173f5d14df0d"
PATCH_GZIP_B64 = """..."""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/tmp/wave124")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave124-silu-rowcta-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave124.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave124-sm75.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("sm75 ptxas resource gate failed")
    base_resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda.ptx",
                         "-o", ROOT / "wave124-base.cubin"], check=False)
    save("ptxas-base.log", base_resource)
    entry = re.search(
        r"Compiling entry function 'gl_silu_mul_quantize_q8_stacked_rowcta_nostore'.*?"
        r"(?=Compiling entry function|\Z)",
        base_resource.stderr,
        re.S,
    )
    if (base_resource.returncode or not entry
            or "0 bytes spill stores, 0 bytes spill loads" not in entry.group(0)):
        raise RuntimeError(f"Wave124 row-CTA resource gate failed: {entry.group(0) if entry else 'missing'}")
    (RESULTS / "wave124-rowcta-resource.txt").write_text(entry.group(0), encoding="utf-8")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave124_silu_rowcta", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave124_silu_rowcta"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "production-ab"
    records = []
    summaries = []
    for invocation in ["a", "b"]:
        measured = run([exe, MODEL, invocation], cwd=TREE, env=prod_env, check=False)
        save(f"production-{invocation}.log", measured)
        if measured.returncode:
            raise RuntimeError(f"production {invocation} failed")
        for raw in re.findall(r"\[wave124-sample\]\s*(\{[^\n]+\})", measured.stdout):
            records.append(json.loads(raw))
        found = re.search(r"\[wave124-summary\]\s*(\{[^\n]+\})", measured.stdout)
        if not found:
            raise RuntimeError(f"production {invocation} summary missing")
        summaries.append(json.loads(found.group(1)))
    if len(records) != 80:
        raise RuntimeError(f"expected 80 production samples, saw {len(records)}")
    if {r["oracle_token"] for r in records} != {3323}:
        raise RuntimeError("oracle drift in production samples")

    retained = [r["prefill_ms"] for r in records if r["arm"] == "retained"]
    candidate = [r["prefill_ms"] for r in records if r["arm"] == "candidate"]
    if len(retained) != len(candidate) or len(retained) != 40:
        raise RuntimeError(f"bad arm sample counts: retained={len(retained)} candidate={len(candidate)}")
    retained_sorted = sorted(retained)
    candidate_sorted = sorted(candidate)
    retained_median = retained_sorted[len(retained_sorted) // 2]
    candidate_median = candidate_sorted[len(candidate_sorted) // 2]
    combined_speedup = retained_median / candidate_median
    retention_gate = {
        "minimum_combined_speedup": 1.02,
        "combined_speedup_pass": combined_speedup >= 1.02,
        "both_invocations_positive": all(s["speedup"] > 1.0 for s in summaries),
    }
    retention_gate["passed"] = all([
        retention_gate["combined_speedup_pass"],
        retention_gate["both_invocations_positive"],
    ])
    if not retention_gate["passed"]:
        raise RuntimeError(f"Wave124 production gate failed: {retention_gate}, summaries={summaries}")

    phase = "candidate-profile"
    prof_env = {**prod_env, "GLCUDA_TELEMETRY": "1"}
    prof = run([exe, MODEL, "profile"], cwd=TREE, env=prof_env, check=False)
    save("candidate-profile.log", prof)
    if prof.returncode or "[wave124-profile]" not in prof.stdout:
        raise RuntimeError("candidate production profile failed")
    profile = json.loads(re.search(r"\[wave124-profile\]\s*(\{[^\n]+\})", prof.stdout).group(1))
    stages = [json.loads(x) for x in re.findall(r"\[wave124-stage\]\s*(\{[^\n]+\})", prof.stdout)]
    expected_names = [
        "qkv",
        "attn_norm",
        "attn_kv_write",
        "attention",
        "attn_out_quant",
        "ffn_down",
        "ffn_gate_up",
        "attn_out",
        "lm_head",
        "ffn_residual_norm_quant",
        "ffn_silu_quant",
        "ffn_residual_add",
    ]
    if [s["name"] for s in stages] != expected_names or profile["oracle_token"] != 3323:
        raise RuntimeError(f"profile contract failed: {profile}, {stages}")
    stage_sum = sum(x["total_ms"] for x in stages)
    ranked = sorted(
        [
            {**s,
             "share_of_gpu_total": s["total_ms"] / profile["gpu_prefill_ms"],
             "share_of_stage_sum": s["total_ms"] / stage_sum if stage_sum else 0.0}
            for s in stages
        ],
        key=lambda s: -s["total_ms"],
    )
    by_name = {x["name"]: x for x in stages}
    summary = {
        "wave": 123,
        "candidate": "stacked_silu_rowcta",
        "gpu": fields,
        "model": model_meta,
        "production_ab": {
            "samples_per_arm": len(retained),
            "retained_median_ms": retained_median,
            "candidate_median_ms": candidate_median,
            "retained_tps": 244000.0 / retained_median,
            "candidate_tps": 244000.0 / candidate_median,
            "speedup": retained_median / candidate_median,
            "all_candidate_deltas_positive": all(c < r for c, r in zip(candidate, retained)),
            "invocation_summaries": summaries,
        },
        "candidate_profile": profile,
        "candidate_stages": stages,
        "candidate_ranked_stages": ranked,
        "candidate_stage_sum_ms": stage_sum,
        "candidate_attention_ms": sum(by_name[n]["total_ms"] for n in [
            "qkv", "attn_norm", "attn_kv_write", "attention", "attn_out_quant", "attn_out"
        ]),
        "candidate_ffn_ms": sum(by_name[n]["total_ms"] for n in [
            "ffn_down", "ffn_gate_up", "ffn_residual_norm_quant", "ffn_silu_quant",
            "ffn_residual_add"
        ]),
        "retention_gate": retention_gate,
        "retention_authority": True,
        "target_15000_tps_achieved": 244000.0 / candidate_median >= 15000,
    }
    report = [
        "# Wave 124 stacked SiLU row-CTA",
        "",
        f"- Production A/B samples: {len(retained)} per arm, counterbalanced a+b",
        f"- Retained median: {retained_median:.6f} ms = {summary['production_ab']['retained_tps']:.1f} tok/s",
        f"- Candidate median: {candidate_median:.6f} ms = {summary['production_ab']['candidate_tps']:.1f} tok/s",
        f"- Speedup: {summary['production_ab']['speedup']:.4f}x",
        f"- Fixed retention gate passed: {retention_gate['passed']}",
        f"- Target 15k reached: {summary['target_15000_tps_achieved']}",
        f"- Candidate profile GPU: {profile['gpu_prefill_ms']:.6f} ms = {profile['gpu_prefill_tps']:.1f} tok/s",
        "",
        "| candidate stage | ms | share of GPU total | calls | bytes read | macs |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for s in ranked:
        report.append(
            f"| `{s['name']}` | {s['total_ms']:.6f} | {100.0 * s['share_of_gpu_total']:.2f}% | "
            f"{s['calls']} | {s['bytes_read']} | {s['macs']} |"
        )
    (RESULTS / "production-records.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    (RESULTS / "wave124-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (RESULTS / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("WAVE124_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
